# Individual TF-IDF word clouds (majority-allegiance rule)

This notebook regenerates the word clouds as **one PNG per entity** so the
website can show them in tabs instead of a single composite grid.

Two changes from `House_TF_IDF.ipynb` / `MainCharacters_TF_IDF.ipynb`:

1. **Majority-allegiance rule.** Each character's allegiance is the entry
   with the highest total membership across the whole dataset (ties broken
   by the wiki's listed order), with the three Baratheon labels merged.
   This matches `graph_analysis_explainer.ipynb` and the website, replacing
   the older first-listed / region-inference rule.
2. **Individual output.** One file per house and per character, written to
   `website/figures/word_clouds/house_<slug>.png` and `char_<slug>.png`.

The tokenisation, collocation, and TF-IDF stages are kept identical to the
canonical notebooks. Run top to bottom; it prints the ordered slug list to
paste into the website tab markup.

In [1]:
import math
import os
import re
import string
import random
from collections import Counter
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, MWETokenizer
from nltk.collocations import BigramCollocationFinder, BigramAssocMeasures
from wordcloud import WordCloud

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

TOP_HOUSES = 14   # how many largest allegiances get a cloud
OUT_DIR = Path('../website/figures/word_clouds')
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAIN_CHARACTERS = [
    'Jon_Snow', 'Cersei_Lannister', 'Daenerys_Targaryen', 'Tyrion_Lannister',
    'Petyr_Baelish', 'Theon_Greyjoy', 'Melisandre', 'Walder_Frey',
    'Davos_Seaworth', 'Robb_Stark',
]


def slug(s):
    return re.sub(r'[^a-z0-9]+', '_', s.lower()).strip('_')

In [2]:
df = pd.read_csv('../csvs/characters_enriched_v3.csv').fillna('')
bios_df = pd.read_csv('../csvs/characters_bio.csv').fillna('')
bios = dict(zip(bios_df['ID'], bios_df['bio']))

if os.path.exists('../csvs/characters_quotes.csv'):
    quotes_df = pd.read_csv('../csvs/characters_quotes.csv').fillna('')
    quotes_df = quotes_df[quotes_df['speaker_id'].str.len() > 0]
    quotes = quotes_df.groupby('speaker_id')['quote'].apply(lambda s: ' '.join(s)).to_dict()
    print(f'Loaded {len(quotes_df)} quotes for {len(quotes)} speakers')
else:
    quotes = {}
    print('characters_quotes.csv not found - bios only')

name_by_id = dict(zip(df['ID'], df['name']))
valid_ids = set(df['ID'])
MAIN_CHARACTERS = [c for c in MAIN_CHARACTERS if c in valid_ids]

# --- Majority-allegiance rule (matches graph_analysis_explainer.ipynb) ---
BARATHEON_VARIANTS = {
    'House_Baratheon_of_Dragonstone': 'House_Baratheon',
    "House_Baratheon_of_King's_Landing": 'House_Baratheon',
}


def normalize_baratheon(s):
    seen, out = set(), []
    for p in (x.strip() for x in s.split(';') if x.strip()):
        p = BARATHEON_VARIANTS.get(p, p)
        if p not in seen:
            seen.add(p)
            out.append(p)
    return ';'.join(out)


df['allegiance'] = df['allegiance'].apply(normalize_baratheon)

multi_counts = Counter()
for a in df['allegiance']:
    for h in (x.strip() for x in a.split(';') if x.strip()):
        multi_counts[h] += 1


def pick_primary(allegiance_str):
    entries = [h.strip() for h in allegiance_str.split(';') if h.strip()]
    if not entries:
        return ''
    return max(entries, key=lambda h: (multi_counts.get(h, 0), -entries.index(h)))


primary_house = {cid: pick_primary(a) for cid, a in zip(df['ID'], df['allegiance'])}

house_counts = Counter(h for h in primary_house.values() if h)
top_houses = [h for h, _ in house_counts.most_common(TOP_HOUSES)]
print(f'Top-{TOP_HOUSES} allegiances by majority rule:')
for h, k in house_counts.most_common(TOP_HOUSES):
    print(f'  {k:>4}  {h}')

# Character-name filter: any token appearing in any character display name
character_name_tokens = set()
for nm in df['name']:
    for part in str(nm).lower().split():
        c = part.strip(string.punctuation)
        if c and c.isalpha() and len(c) > 2:
            character_name_tokens.add(c)
print(f'Character-name filter: {len(character_name_tokens)} tokens')

Loaded 4275 quotes for 505 speakers
Top-14 allegiances by majority rule:
   236  House_Targaryen
   176  Night's_Watch
   150  Citadel
   138  House_Frey
   133  House_Stark
   108  House_Lannister
    97  Blacks
    96  House_Baratheon
    77  House_Greyjoy
    75  Faith_of_the_Seven
    69  Kingsguard
    58  House_Tyrell
    53  Mance_Rayder
    52  House_Martell
Character-name filter: 3033 tokens


In [3]:
def base_tokenize(text):
    return [w for w in word_tokenize(text.lower())
            if w not in stop_words
            and w not in string.punctuation
            and w.isalpha()
            and w not in character_name_tokens
            and len(w) > 2]


# Collocations discovered on the whole corpus (course week 7: chi-squared).
all_tokens = []
for cid in df['ID']:
    text = bios.get(cid, '') + ' ' + quotes.get(cid, '')
    if text.strip():
        all_tokens.extend(base_tokenize(text))

finder = BigramCollocationFinder.from_words(all_tokens)
finder.apply_freq_filter(30)
collocations = finder.nbest(BigramAssocMeasures().chi_sq, 100)
mwe = MWETokenizer(collocations, separator='_')
print(f'{len(collocations)} significant bigrams; e.g. '
      f'{["_".join(b) for b in collocations[:8]]}')


def tokenize(text):
    return mwe.tokenize(base_tokenize(text))


# Per-house token pools (majority rule) and per-character token pools.
house_tokens = {h: [] for h in top_houses}
for cid, house in primary_house.items():
    if house in house_tokens:
        text = bios.get(cid, '') + ' ' + quotes.get(cid, '')
        if text.strip():
            house_tokens[house].extend(tokenize(text))

char_tokens = {}
for cid in MAIN_CHARACTERS:
    text = bios.get(cid, '') + ' ' + quotes.get(cid, '')
    char_tokens[cid] = tokenize(text) if text.strip() else []

for h in top_houses:
    print(f'  {h:32s} {len(house_tokens[h]):>7} tokens')

100 significant bigrams; e.g. ['television_adaptation', 'moat_cailin', 'game_thrones', 'casterly_rock', 'vaes_dothrak', 'deepwood_motte', 'seastone_chair', 'adaptation_game']
  House_Targaryen                    34404 tokens
  Night's_Watch                      20110 tokens
  Citadel                             8034 tokens
  House_Frey                          6745 tokens
  House_Stark                        25261 tokens
  House_Lannister                    23908 tokens
  Blacks                              5625 tokens
  House_Baratheon                    14834 tokens
  House_Greyjoy                       9896 tokens
  Faith_of_the_Seven                  4168 tokens
  Kingsguard                          5664 tokens
  House_Tyrell                        2697 tokens
  Mance_Rayder                        3666 tokens
  House_Martell                       6202 tokens


In [4]:
def tfidf(doc_tokens):
    """doc_tokens: {label: [tokens]} -> {label: {term: tfidf}}."""
    tf = {}
    for label, toks in doc_tokens.items():
        if not toks:
            tf[label] = {}
            continue
        cnt = Counter(toks)
        total = len(toks)
        tf[label] = {w: c / total for w, c in cnt.items()}
    N = len(doc_tokens)
    dfreq = Counter()
    for toks in doc_tokens.values():
        for w in set(toks):
            dfreq[w] += 1
    idf = {w: math.log(N / dfreq[w]) for w in dfreq}
    return {label: {w: t * idf.get(w, 0) for w, t in tfv.items()}
            for label, tfv in tf.items()}


tfidf_house = tfidf(house_tokens)
tfidf_char = tfidf(char_tokens)

biggest = top_houses[0]
print(f'Top-10 TF-IDF terms for {biggest}:')
for w, s in sorted(tfidf_house[biggest].items(), key=lambda x: -x[1])[:10]:
    print(f'  {s:.5f}  {w}')

Top-10 TF-IDF terms for House_Targaryen:
  0.00568  unsullied
  0.00514  khalasar
  0.00401  meereen
  0.00307  bloodriders
  0.00269  yunkai
  0.00243  griff
  0.00237  astapor
  0.00209  khal
  0.00198  drogon
  0.00176  dany


In [5]:
karma_colors = [
    '#2A4A7F', '#3D6FA8', '#5B8DB8', '#C9A84C', '#D4B86A',
    '#E8C97A', '#F5E4A0', '#7BA3C8', '#A07830', '#4A7FA0',
]
karma_cmap = LinearSegmentedColormap.from_list('karma', karma_colors)


def karma_color_func(word, **kwargs):
    r, g, b, _ = karma_cmap(random.Random(word).random())
    return f'rgb({int(r*255)}, {int(g*255)}, {int(b*255)})'


def render(freqs, out_path, title):
    fig, ax = plt.subplots(figsize=(12, 7))
    fig.patch.set_facecolor('#0a0c14')
    ax.set_facecolor('#12151f')
    if freqs:
        wc = WordCloud(width=1200, height=700, background_color='#12151f',
                       color_func=karma_color_func, max_words=120,
                       prefer_horizontal=0.9)
        ax.imshow(wc.generate_from_frequencies(freqs), interpolation='bilinear')
    else:
        ax.text(0.5, 0.5, 'No tokens', ha='center', va='center',
                color='#C9A84C', transform=ax.transAxes)
    ax.set_title(title, fontsize=14, pad=8, color='#E8C97A')
    ax.axis('off')
    fig.savefig(out_path, dpi=200, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close(fig)


house_manifest, char_manifest = [], []

for h in top_houses:
    fname = f'house_{slug(h)}.png'
    render(tfidf_house.get(h, {}), OUT_DIR / fname,
           f"{h.replace('_', ' ')}  (n={house_counts[h]})")
    house_manifest.append((fname, h.replace('_', ' ')))

for cid in MAIN_CHARACTERS:
    fname = f'char_{slug(cid)}.png'
    render(tfidf_char.get(cid, {}), OUT_DIR / fname, name_by_id[cid])
    char_manifest.append((fname, name_by_id[cid]))

print(f'Wrote {len(house_manifest) + len(char_manifest)} clouds to {OUT_DIR}\n')
print('=== HOUSE TABS (order :: file :: label) ===')
for i, (f, lbl) in enumerate(house_manifest, 1):
    print(f'  {i:>2}  {f}  ::  {lbl}')
print('\n=== CHARACTER TABS ===')
for i, (f, lbl) in enumerate(char_manifest, 1):
    print(f'  {i:>2}  {f}  ::  {lbl}')

Wrote 24 clouds to ../website/figures/word_clouds

=== HOUSE TABS (order :: file :: label) ===
   1  house_house_targaryen.png  ::  House Targaryen
   2  house_night_s_watch.png  ::  Night's Watch
   3  house_citadel.png  ::  Citadel
   4  house_house_frey.png  ::  House Frey
   5  house_house_stark.png  ::  House Stark
   6  house_house_lannister.png  ::  House Lannister
   7  house_blacks.png  ::  Blacks
   8  house_house_baratheon.png  ::  House Baratheon
   9  house_house_greyjoy.png  ::  House Greyjoy
  10  house_faith_of_the_seven.png  ::  Faith of the Seven
  11  house_kingsguard.png  ::  Kingsguard
  12  house_house_tyrell.png  ::  House Tyrell
  13  house_mance_rayder.png  ::  Mance Rayder
  14  house_house_martell.png  ::  House Martell

=== CHARACTER TABS ===
   1  char_jon_snow.png  ::  Jon Snow
   2  char_cersei_lannister.png  ::  Cersei Lannister
   3  char_daenerys_targaryen.png  ::  Daenerys Targaryen
   4  char_tyrion_lannister.png  ::  Tyrion Lannister
   5  char_pety